ML LAB 
17 OCTOBER

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree, _tree
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score)
from IPython.display import Image,display
import joblib


In [2]:
dataset_path="Customer_Churn.csv"
df=pd.read_csv(dataset_path)
print(f'total samples(rows): {df.shape}')
df.head()

total samples(rows): (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.shape

(7043, 21)

In [4]:
print("dataset information")
print(df.info())
#target class distribution
print(df['Churn'].value_counts(dropna=False))

dataset information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043

data cleaning

In [5]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_total_charges = df['TotalCharges'].isnull().sum()
print("missing values in TotalCharges:", missing_total_charges)

median_val=df['TotalCharges'].median()
df['TotalCharges']=df['TotalCharges'].fillna(median_val)
print(f'Imputed with median({median_val:.2f}). remaining nulls:{df.isnull().sum()}')#.sum we add 2 times a new ftn so there is no null all are zeros

missing values in TotalCharges: 11
Imputed with median(1397.47). remaining nulls:customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [6]:
df = df.drop(columns=['customerID'], errors='ignore')
print(f'After dropping customerID dataset shape: {df.shape}')

After dropping customerID dataset shape: (7043, 20)


In [7]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("target variable mapped distribution:")
print(df['Churn'].value_counts())

target variable mapped distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [9]:
X_raw=df.drop(columns=['Churn'], errors='ignore')
y=df['Churn']
X=pd.get_dummies(X_raw, drop_first=True)
print(f'feature matrix X shape: {X.shape}')
print(f'target vector y shape: {y.shape}')
print(f'total encoded columns: {X.shape[1]}')

feature matrix X shape: (7043, 30)
target vector y shape: (7043,)
total encoded columns: 30


In [11]:
print(X)

      SeniorCitizen  tenure  MonthlyCharges  TotalCharges  gender_Male  \
0                 0       1           29.85         29.85        False   
1                 0      34           56.95       1889.50         True   
2                 0       2           53.85        108.15         True   
3                 0      45           42.30       1840.75         True   
4                 0       2           70.70        151.65        False   
...             ...     ...             ...           ...          ...   
7038              0      24           84.80       1990.50         True   
7039              0      72          103.20       7362.90        False   
7040              0      11           29.60        346.45        False   
7041              1       4           74.40        306.60         True   
7042              0      66          105.65       6844.50         True   

      Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0            True           False             False   
1

TRAINING TESTING

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
#stratify for same distribution of target class in both train and test sets, same accuracy for testing and training 
print("Data Summary")
print(f"Training features: {X_train.shape}")
print(f"Testing features: {X_test.shape}")
print(f"Training features: {y_train.shape}")
print(f"Testing features: {y_test.shape}")

Data Summary
Training features: (5634, 30)
Testing features: (1409, 30)
Training features: (5634,)
Testing features: (1409,)


In [18]:
#bring dataset values into a single range 
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
print("feature scaling ")
print(f"Training features mean: {np.round(X_train_scaled.mean(), 2)}")
print(f"Testing features mean: {np.round(X_test_scaled.mean(), 2)}")

feature scaling 
Training features mean: -0.0
Testing features mean: -0.01


In [19]:
model=DecisionTreeClassifier(
    criterion='gini',
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
print('decision tree model:', model)

decision tree model: DecisionTreeClassifier(max_depth=10, min_samples_leaf=5, min_samples_split=10,
                       random_state=42)


In [21]:
model.fit(X_train_scaled, y_train)
train_accuracy = model.score(X_train_scaled, y_train)
print('model training completed successfully')
print(f'training accuracy: {train_accuracy * 100:.2f}%')

model training completed successfully
training accuracy: 85.23%


In [22]:
print('MODEL HYPERPARAMETERS')
for param, value in model.get_params().items():
    print(f'{param:25s}: {value}')

print(' LEARNED TREE ATTRIBUTES ')
print(f'Tree Maximum Depth          : {model.get_depth()}')
print(f'Total Number of Leaves      : {model.get_n_leaves()}')
print(f'Number of Target Classes    : {model.n_classes_}')

print('TOP 5 FEATURE IMPORTANCES ')
feature_importances = pd.Series(model.feature_importances_, index=X.columns)
top_5_features = feature_importances.nlargest(5)
for feature_name, importance_score in top_5_features.items():
    print(f'{feature_name:32s}: {importance_score:.4f}')

MODEL HYPERPARAMETERS
ccp_alpha                : 0.0
class_weight             : None
criterion                : gini
max_depth                : 10
max_features             : None
max_leaf_nodes           : None
min_impurity_decrease    : 0.0
min_samples_leaf         : 5
min_samples_split        : 10
min_weight_fraction_leaf : 0.0
monotonic_cst            : None
random_state             : 42
splitter                 : best
 LEARNED TREE ATTRIBUTES 
Tree Maximum Depth          : 10
Total Number of Leaves      : 314
Number of Target Classes    : 2
TOP 5 FEATURE IMPORTANCES 
tenure                          : 0.2976
InternetService_Fiber optic     : 0.2213
TotalCharges                    : 0.1420
MonthlyCharges                  : 0.1123
PaymentMethod_Electronic check  : 0.0276
